In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────
# Installs are quiet (-q) so the output stays readable on a projector.
!pip install -q google-genai pandas matplotlib

import os, json, time                    # standard library
from google import genai                 # the SDK
from google.genai import types           # config and content types
import pandas as pd
import matplotlib.pyplot as plt
import functools
import random
from datetime import datetime

# ── Your API key ──────────────────────────────────────────────────────
# The key lives in Colab Secrets, never in the notebook. If this cell
# fails, that is almost always why.
try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
    if not API_KEY:
        raise ValueError("empty")
except Exception:
    raise SystemExit(
        "\n" + "=" * 68 +
        "\nNo API key found.\n"
        "\n  1. Click the KEY icon in the left sidebar of Colab."
        "\n  2. Click 'Add new secret'."
        "\n  3. Name it exactly:  GEMINI_API_KEY"
        "\n  4. Paste your key from aistudio.google.com"
        "\n  5. Turn ON 'Notebook access' for this notebook."
        "\n  6. Run this cell again."
        "\n" + "=" * 68
    )

client = genai.Client(api_key=API_KEY)

MODEL = "gemini-2.5-flash-lite"          # fast and cheap; what we use all week
EMBED_MODEL = "gemini-embedding-001"    # free tier, which is what makes Day 2 possible

print("Ready. Model:", MODEL)

# Bring in your own work from the last two days: paste your ask(),
# hybrid_search() and run_agent() into this notebook, or re-run those
# cells here. Today is about measuring YOUR project, not a toy.

# Day 4 — Production and the numbers

**By the end of this notebook** you will have a table of your own numbers:
latency per request, tokens in and out, cost, and a **measured** cache hit
rate — and a per-user-per-month figure you can say out loud in a meeting.

That table goes in your presentation tomorrow. Pairs who show measured
numbers read as engineers. Pairs who say "it feels fast" do not.

In [ ]:
# A retry decorator with exponential backoff. GIVEN COMPLETE.
#
# The important design decision is not the retrying — it is deciding WHAT to
# retry. A 429 will probably succeed in two seconds. A 400 will fail
# identically forever, and retrying it wastes your budget and the user's time.

RETRYABLE = ("429", "500", "502", "503", "504", "RESOURCE_EXHAUSTED",
             "UNAVAILABLE", "DEADLINE_EXCEEDED")


def retry(tries=3, base=1.0):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            for attempt in range(tries):
                try:
                    return fn(*args, **kwargs)
                except Exception as e:
                    text = str(e)
                    retryable = any(code in text for code in RETRYABLE)
                    if not retryable or attempt == tries - 1:
                        raise
                    wait = base * (2 ** attempt)      # 1s, 2s, 4s
                    print(f"  retryable error, waiting {wait}s: {text[:60]}")
                    time.sleep(wait)
        return wrapper
    return decorator


@retry(tries=3)
def generate(prompt, model=None):
    return client.models.generate_content(model=model or MODEL, contents=prompt)


print(generate("Say OK in one word.").text)

In [ ]:
# Prove the retry actually fires. This function fails twice, then succeeds.
attempts = {"n": 0}


@retry(tries=4, base=0.5)
def flaky():
    attempts["n"] += 1
    if attempts["n"] < 3:
        raise RuntimeError("503 UNAVAILABLE (simulated)")
    return f"succeeded on attempt {attempts['n']}"


print(flaky())

# And prove it does NOT retry something pointless.
attempts["n"] = 0


@retry(tries=4, base=0.5)
def bad_request():
    attempts["n"] += 1
    raise ValueError("400 INVALID_ARGUMENT (simulated)")


try:
    bad_request()
except ValueError:
    print("gave up after", attempts["n"], "attempt(s) - correct, 400 is not retryable")

In [ ]:
# Streaming vs non-streaming, side by side, timed.
PROMPT = "Explain retrieval-augmented generation to a manager in about 150 words."

t0 = time.time()
full = client.models.generate_content(model=MODEL, contents=PROMPT)
non_streaming_total = time.time() - t0
print(f"NON-STREAMING: first word after {non_streaming_total:.2f}s, "
      f"total {non_streaming_total:.2f}s")

t0 = time.time()
first_token_at = None
for chunk in client.models.generate_content_stream(model=MODEL, contents=PROMPT):
    if first_token_at is None:
        first_token_at = time.time() - t0
streaming_total = time.time() - t0

print(f"STREAMING:     first word after {first_token_at:.2f}s, "
      f"total {streaming_total:.2f}s")

### Streaming does not make it faster. It makes it *feel* faster.

Look at the two totals above — they are within noise of each other. Total
generation time did not change by a millisecond.

What changed is **time to first token**: from several seconds of blank screen
to a few hundred milliseconds. Users read that as speed, and it is one of the
highest-value changes you can make to an AI product.

It is a perception fix, and perception is most of user experience. Just be
honest with yourself about which problem you solved: if your p95 latency is
too high, streaming hides it rather than fixing it.

In [ ]:
# A cache keyed on the prompt. Twelve lines, best value in the file.
import hashlib

CACHE = {}
stats = {"hits": 0, "misses": 0, "time_saved": 0.0}


def cached_generate(prompt, **kw):
    # Key on the prompt AND the settings — a different temperature is a
    # different question and must not share a cache entry.
    key = hashlib.sha256((prompt + json.dumps(kw, sort_keys=True)).encode()).hexdigest()

    if key in CACHE:
        stats["hits"] += 1
        stats["time_saved"] += CACHE[key]["latency"]
        return CACHE[key]["text"]

    stats["misses"] += 1
    t0 = time.time()
    r = generate(prompt)
    latency = time.time() - t0

    CACHE[key] = {"text": r.text, "latency": latency}
    return r.text


# In production this is Redis with a time-to-live, not a dict. A cache with
# no expiry serves yesterday's policy after it changed — that is a
# correctness bug, not a performance detail.
print(cached_generate("What is a token, in one sentence?")[:80])
print(cached_generate("What is a token, in one sentence?")[:80], "(cached)")
print(stats)

In [ ]:
# 20 queries with 8 repeats. Measure the hit rate and the time saved.
BASE_QUESTIONS = [
    "How many annual leave days does grade 11 get?",
    "How do I request a training programme?",
    "How quickly must a data breach be reported?",
    "What is the procurement threshold for a competitive process?",
    "Can I work remotely three days a week?",
    "What multi-factor authentication is required?",
    "Who approves overtime?",
    "What is the sick leave entitlement?",
    "How long are records retained?",
    "What is the IT response time for a priority 2 issue?",
    "Are gifts allowed?",
    "Can I use personal cloud storage for work files?",
]

# 12 unique + 8 repeats = 20 queries.
queries = BASE_QUESTIONS + random.sample(BASE_QUESTIONS, 8)
random.shuffle(queries)

CACHE.clear()
stats.update({"hits": 0, "misses": 0, "time_saved": 0.0})

t0 = time.time()
for q in queries:
    cached_generate(q)
elapsed = time.time() - t0

total = stats["hits"] + stats["misses"]
print(f"queries       : {total}")
print(f"cache hits    : {stats['hits']}")
print(f"hit rate      : {stats['hits'] / total:.0%}")
print(f"time saved    : {stats['time_saved']:.1f}s")
print(f"wall clock    : {elapsed:.1f}s")

In [ ]:
# TODO ─ The cost formula. This comes BEFORE log_request, which calls it —
#        so the notebook cannot break if someone runs cells out of order.
#
# Look up the CURRENT prices for gemini-2.5-flash-lite. Prices are quoted
# per MILLION tokens, and input and output are priced differently.
# Reading a pricing page is a skill; it changes every few months.

PRICE_PER_1M_INPUT = 0.0        # ← TODO: put the real figure here
PRICE_PER_1M_OUTPUT = 0.0       # ← TODO: put the real figure here


def cost_of(prompt_tokens, output_tokens):
    # ← TODO (1 line): return the cost of this single request in dollars.
    #   Remember: the prices above are per 1,000,000 tokens.
    return 0.0


# Sanity check: a 2,000-in / 400-out request should be a fraction of a cent.
print(cost_of(2000, 400))

In [ ]:
# log_request() — one row per call. This is the whole of observability
# at small scale, and it is what the paid tools do for you at large scale.
LOG = []


def log_request(prompt, response, latency, cached=False, model=None):
    u = getattr(response, "usage_metadata", None)
    prompt_tokens = getattr(u, "prompt_token_count", 0) if u else 0
    output_tokens = getattr(u, "candidates_token_count", 0) if u else 0

    LOG.append({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "model": model or MODEL,
        "prompt_tokens": prompt_tokens,
        "output_tokens": output_tokens,
        "latency": round(latency, 3),
        "cached": cached,
        "cost": cost_of(prompt_tokens, output_tokens),
    })


print("Logging ready. Cost per request will read 0.0 until you fill in the "
      "two prices in the cell above.")

In [ ]:
# 15 mixed queries, logged, into a DataFrame.
LOG.clear()

for q in random.sample(BASE_QUESTIONS, 12) + BASE_QUESTIONS[:3]:
    t0 = time.time()
    r = generate(q)
    log_request(q, r, time.time() - t0)

df = pd.DataFrame(LOG)
print(df.head())
print()
print(df[["prompt_tokens", "output_tokens", "latency", "cost"]].describe())
print()
print(f"median latency : {df['latency'].median():.2f}s")
print(f"p95 latency    : {df['latency'].quantile(0.95):.2f}s")
print(f"total cost     : ${df['cost'].sum():.6f}")

In [ ]:
# Two plots: where the latency sits, and how cost accumulates.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["latency"], bins=10)
axes[0].axvline(df["latency"].median(), linestyle="--")
axes[0].set_title("Latency distribution")
axes[0].set_xlabel("seconds")
axes[0].set_ylabel("requests")

axes[1].plot(range(1, len(df) + 1), df["cost"].cumsum(), marker="o")
axes[1].set_title("Cumulative cost")
axes[1].set_xlabel("request number")
axes[1].set_ylabel("dollars")

plt.tight_layout()
plt.show()

# The median is what a typical user waits. The right-hand tail is what
# people complain about. Optimise the median, but design for the tail.

### TODO — from your table, work out the real numbers

Using the DataFrame you just produced:

**Cost per query (mean):**

> _your answer here_

**At 4 queries per user per working day, 22 working days: cost per user per
month?**

> _your answer here_

**For 500 users, what is the monthly bill?**

> _your answer here_

**Your measured cache hit rate was ____%. What does that do to the number
above?**

> _your answer here_

**And in Arabic — apply the 2.5x token multiplier from Day 1. What is the
monthly bill now?**

> _your answer here_

In [ ]:
# A fallback chain: primary model → cheaper model → canned response.
CANNED = ("The assistant is unavailable right now. The policy documents "
          "are on the intranet under HR → Policies.")


def ask_with_fallback(prompt, models=None):
    for model in (models or [MODEL, MODEL]):     # replace the second with a
        try:                                     # genuinely cheaper model
            return generate(prompt, model=model).text
        except Exception as e:
            print(f"  {model} failed ({str(e)[:50]}), falling back")
    return CANNED


# Break the primary on purpose and watch the fallback engage.
print(ask_with_fallback("What is a token?", models=["does-not-exist-model", MODEL])[:120])
print()
print("Both broken:")
print(ask_with_fallback("What is a token?", models=["nope-1", "nope-2"]))

# Your users will forgive a degraded answer. They will not forgive a page
# that hangs.

## Reflection

Fill these in before you close the notebook. This is what I check when I come round.

**What is your median latency, and your p95? Which one will your users complain about?**

> _your answer here_

**What is your cost per user per month, and what did you assume to get there?**

> _your answer here_

**Your cache hit rate was what? Which single change would raise it most?**

> _your answer here_

**Which of retries, caching, streaming or fallbacks would you add to your project first, and why that one?**

> _your answer here_

## If this breaks

The three most likely failures, and what to do about each.

| Symptom | Cause | Fix |
|---|---|---|
| `cost` column is all zeros | The TODO in the cost cell is still returning 0.0 | Fill in the two price constants and the return line, then re-run the logging cell |
| `NameError: cost_of` | The cost cell was defined after `log_request` but never run | Run cells in order; `log_request` calls `cost_of` at call time, not at definition time |
| Plots do not appear | Matplotlib backend, or the cell ran before `df` existed | Re-run the DataFrame cell first; in Colab `plt.show()` is enough, no magic needed |